# Module 10 — Fine-Tuning Protein Structure Models with LoRA

## Learning Objectives
- [ ] Understand why fine-tuning beats training from scratch for protein ML
- [ ] Implement LoRA from scratch and inject it into a Pairformer
- [ ] Build a complete ΔΔG mutation effect prediction pipeline
- [ ] Compare frozen / LoRA / full fine-tune modes by trainable parameter count
- [ ] Implement warmup + cosine LR schedule from scratch
- [ ] Run mixed-precision training (bfloat16) and explain why it works
- [ ] Save a LoRA adapter checkpoint (10 MB vs 2 GB full model)
- [ ] Understand the TCR-pMHC capstone goal: CDR3β RMSD < 1.5 Å

## Beginner Teaching Frame

**Who should fully work through this notebook:** advanced students after the core path. Others should study it conceptually first.

**How to study it on a first pass:**
- understand the task framing, adaptation strategy, and evaluation logic
- treat the Pairformer details as a second-pass target
- separate the scientific question from the engineering machinery

**Common traps:** conflating didactic mock implementations with production systems and chasing every code detail too early.


## Canonical Resource Upgrade

Recommended references:
- [OpenFold](https://github.com/aqlaboratory/openfold)
- [Hugging Face PEFT docs](https://huggingface.co/docs/peft)
- [ProteinGym](https://github.com/OATML-Markslab/ProteinGym)


## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | ALL modules — this is the capstone. Minimum: 07/01–07/04 (AF3), 05/01 (transformers), 09/01 (diagnostics) |
| 📍 You Are Here | Module 10/01 — Fine-Tuning Protein Structure Models (Capstone) |
| ➡️ Next Steps | Research / Industry — this is the final notebook; next step is applying to real datasets or jobs |
| 🏁 Goal | Fine-tune ESM2 with LoRA on ProteinGym; compare frozen/LoRA/full modes; achieve Spearman r > 0.4 on held-out mutations |

### 🆕 From Scratch? Start Here:
1. [PEFT library docs](https://huggingface.co/docs/peft) — LoRA configuration and training in HuggingFace
2. [LoRA paper (arXiv 2106.09685)](https://arxiv.org/abs/2106.09685) — 4-page essential read
3. 07/01–07/04 (this repo) — AF3 architecture to understand what you are fine-tuning
4. 09/01 (this repo) — model diagnostics to evaluate fine-tuning
5. This notebook — fine-tuning capstone
**Time:** 20–30h | **Difficulty:** ⭐⭐⭐⭐⭐

### 🔗 Cross-References
- Builds on: ALL modules — integrates sequence biology (01), ML fundamentals (04–05), GNNs (06), AF3 architecture (07), algorithms (08), diagnostics (09)
- Used by: Nothing — this is the terminal notebook; outputs are production-ready LoRA adapters
- Capstone: Demonstrates full-stack protein ML competency from data parsing to model deployment

## TL;DR — What This Module Is About

**The situation:** AlphaFold3 and Boltz-2 were trained on millions of protein structures using thousands of GPU-hours. You have hundreds of examples and a laptop.

**The solution:** Fine-tuning. These models have already learned the language of proteins — atom arrangements, residue contacts, geometric constraints. We don't retrain that. We adapt it.

**LoRA (Low-Rank Adaptation):** Instead of updating all 1B weights, we add tiny trainable matrices (rank 8) alongside the frozen Q/K/V projections. Only 0.5% of weights are trainable. The model starts identical to the pretrained version (B initialized to zero) and learns task-specific corrections.

**The task:** Given a wildtype protein sequence and a single point mutation, predict ΔΔG — the change in folding free energy (kcal/mol). Positive = mutation destabilizes the protein. Negative = mutation stabilizes it. This matters for drug design, enzyme engineering, and understanding disease variants.

**Prerequisites:** Module 5 (LoRA basics), Module 7 (Pairformer architecture), Module 9 (training diagnostics).

## Protein Structure Finetuning — Concepts for Beginners

| Term | Plain English |
|------|--------------|
| **LoRA (Low-Rank Adaptation)** | Instead of retraining all weights, inject two small matrices A and B; output = frozen_weight·x + (B·A)·x * scale — only A and B train |
| **rank r** | Controls how many parameters LoRA adds; rank 4 adds 8× fewer params than rank 64 |
| **frozen weights** | Model parameters that don't change during finetuning — only LoRA A/B matrices are updated |
| **Pairformer** | AF3's core module: takes a (L,L) pair matrix and (L,) single matrix, refines both with triangle attention |
| **deltadeltaG (ΔΔG)** | Free energy change upon mutation: ΔΔG = ΔG_mutant − ΔG_wildtype; negative = stabilizing |
| **SKEMPI** | Database of ~7,000 measured binding affinity changes for protein mutations — used as training labels |
| **TCR-pMHC** | T-cell receptor binding to peptide-MHC complex; key cancer immunotherapy target modelled as multi-chain input |
| **head-only finetuning** | Only train the prediction head (last layer) — fastest, but lowest accuracy on new domains |
| **full finetuning** | Train all layers — best accuracy but risks catastrophic forgetting and needs lots of data |
| **train/val split** | Divide data: 80% to learn from, 20% to check generalisation — never train on validation examples |
| **Pearson r** | Correlation coefficient: +1 = perfect agreement, 0 = no relationship, −1 = opposite — used to evaluate ΔΔG prediction |
| **scaffold** | The protein backbone structure that a binding site or function is built on |

## Section 1: Why Fine-Tune?

The three strategies for applying large pretrained models to new tasks:

| Strategy | Data needed | GPU time | Parameters trained | Risk of forgetting |
|---|---|---|---|---|
| **Training from scratch** | Millions of structures | Weeks–months | 100% | N/A (no prior knowledge) |
| **Full fine-tuning** | 1K–100K examples | Hours–days | 100% | High (catastrophic forgetting) |
| **LoRA fine-tuning** | 100–10K examples | Minutes–hours | 0.1–2% | Low (frozen backbone) |

### Why LoRA wins for biology:

1. **Data scarcity is the norm.** FireProtDB has ~4,000 ΔΔG measurements. STCRDab has ~500 TCR-pMHC structures. These are large for biology — tiny for deep learning.

2. **Geometric representations are hard to learn.** The Pairformer took months to learn triangle inequalities and residue co-evolution. We must not corrupt that.

3. **Adapter ecosystem.** One frozen Boltz-2 backbone + many task-specific LoRA adapters (10 MB each) = economical multi-task deployment.

4. **Regularization.** Fewer trainable parameters = less overfitting on small datasets.

In [ ]:
# Section 1: Why fine-tune? — Three strategies
import torch
import torch.nn as nn

print("Fine-tuning Strategies for Protein Structure Models")
print("=" * 60)

strategies = {
    "1. Full Fine-tuning": {
        "What": "Update ALL pretrained weights",
        "Pros": "Maximum flexibility, best performance",
        "Cons": "Expensive (GPU/time), risk of catastrophic forgetting, large checkpoints",
        "When": "Large labeled dataset, enough GPU budget",
    },
    "2. Linear Probing / Head-only": {
        "What": "Freeze backbone, train only classification/regression head",
        "Pros": "Fast, cheap, no catastrophic forgetting",
        "Cons": "Limited adaptation, backbone features may not suit new task",
        "When": "Small dataset, limited compute, backbone already domain-matched",
    },
    "3. LoRA (Parameter-Efficient)": {
        "What": "Inject low-rank matrices into attention layers only",
        "Pros": "~0.1-1% trainable params, same performance as full fine-tuning",
        "Cons": "Need to choose rank r, slightly more complex",
        "When": "Best of both worlds — recommended for protein model fine-tuning",
    },
}

for name, info in strategies.items():
    print(f"\n{name}")
    for k, v in info.items():
        print(f"  {k}: {v}")

# Simulate parameter counts
backbone = nn.TransformerEncoder(
    nn.TransformerEncoderLayer(d_model=256, nhead=8, batch_first=True), num_layers=4)
total = sum(p.numel() for p in backbone.parameters())
print(f"\nExample backbone: {total:,} parameters")
print(f"Head-only trains: ~{total//1000:,} params (0.1%)")
print(f"LoRA rank=8 trains: ~{total//50:,} params (~2%)")
print(f"Full fine-tune trains: {total:,} params (100%)")

## Section 2: The Pairformer — What Makes It Special

Standard transformers operate on a 1D sequence of tokens. The Pairformer operates on two coupled representations:

```
Input tokens (sequence of N amino acids)
    ↓ embedding
s: (N, c_s)      — "what is residue i like?"
z: (N, N, c_z)   — "how do residues i and j relate?"
    ↓ 48 Pairformer blocks
Updated s, z → structure prediction
```

**Why 2D pair representations?** Protein structure is determined by contacts between residues. Whether residues 10 and 47 are close in 3D space cannot be read from a 1D sequence — it requires explicit pairwise reasoning.

**Triangle operations** are the key innovation. The triangle inequality states: if residue i is close to k, and k is close to j, then i and j cannot be arbitrarily far. The triangle multiplicative update enforces this geometric consistency differentiably:

- **Outgoing:** `z[i,j] ← f(z[i,k], z[j,k])`  — update edge (i,j) using all k as intermediaries
- **Incoming:** `z[i,j] ← f(z[k,i], z[k,j])`  — symmetric counterpart

**Key difference from standard transformer:**
- Standard: sequence tokens → 1D attention → updated tokens
- Pairformer: (s, z) → triangle attention/multiplication on z → cross-update s from z → (s', z')

In [ ]:
# Section 2: Pairformer — what makes it special
import torch
import torch.nn as nn
import math

class MiniPairformer(nn.Module):
    """Minimal Pairformer: operates on (L, L, c_z) pair representation.
    Key difference from standard transformer: processes PAIRS of tokens, not tokens.
    """
    def __init__(self, c_z=64, c_s=128, n_heads=4):
        """Initialise MiniPairformer.

Args:
    c_z (int): pair representation channel dimension.
    c_s (int): single representation channel dimension.
    n_heads (int): number of attention heads in triangle attention."""
        super().__init__()
        self.pair_attn = nn.MultiheadAttention(c_z, n_heads, batch_first=True)
        self.pair_norm1 = nn.LayerNorm(c_z)
        self.pair_norm2 = nn.LayerNorm(c_z)
        self.pair_ff = nn.Sequential(
            nn.Linear(c_z, c_z*4), nn.GELU(), nn.Linear(c_z*4, c_z)
        )
        self.single_update = nn.Sequential(
            nn.Linear(c_s, c_s), nn.GELU(), nn.Linear(c_s, c_s)
        )
        self.pair_to_single = nn.Linear(c_z, c_s)

    def forward(self, z, s):
        """Run one forward pass through all Pairformer blocks.

Args:
    z (Tensor): pair representation of shape (B, L, L, c_z).
    s (Tensor): single representation of shape (B, L, c_s).

Returns:
    Tuple[Tensor, Tensor]: updated (z, s) tensors."""
        # z: (B, L, L, c_z) — pair representation
        # s: (B, L, c_s) — single representation
        B, L, _, c_z = z.shape

        # Row-wise attention over pair representation
        z_flat = z.view(B*L, L, c_z)  # treat each row as a sequence
        z_attn, _ = self.pair_attn(self.pair_norm1(z_flat), z_flat, z_flat)
        z_flat = z_flat + z_attn
        z_flat = z_flat + self.pair_ff(self.pair_norm2(z_flat))
        z = z_flat.view(B, L, L, c_z)

        # Update single repr from pair (mean over j dimension)
        z_mean = z.mean(dim=2)  # (B, L, c_z)
        s = s + self.single_update(s) + self.pair_to_single(z_mean)
        return z, s

torch.manual_seed(42)
B, L, c_z, c_s = 2, 16, 64, 128
z = torch.randn(B, L, L, c_z)
s = torch.randn(B, L, c_s)
pairformer = MiniPairformer(c_z=c_z, c_s=c_s)
z_out, s_out = pairformer(z, s)
print(f"Pairformer input:  z={z.shape}, s={s.shape}")
print(f"Pairformer output: z={z_out.shape}, s={s_out.shape}")
print(f"Key: Pairformer processes L² pair tokens, not L sequence tokens")
print(f"Memory: O(L²·c_z) vs O(L·c_s) for standard transformer")

## Section 3: LoRA — Low-Rank Adaptation

### The Math

For a pretrained linear layer with weight **W** ∈ ℝ^(d_out × d_in):

- **W is frozen** — no gradients flow through it
- **LoRA adds:** ΔW = B @ A, where A ∈ ℝ^(r × d_in), B ∈ ℝ^(d_out × r)
- **Effective weight:** W_eff = W + (α/r) × B @ A
- **At initialization:** B = 0, so the model starts **identical** to the pretrained model
- **Training:** Only A and B update — r << d_in, d_out → 99%+ fewer parameters

The scaling factor α/r controls the magnitude of the LoRA correction. Typical values: r=8, α=16 → scale=2.0.

### Parameter Reduction

For a typical protein LM hidden dim d=384:

| Configuration | Parameters | Reduction |
|---|---|---|
| Full W (d=384) | 384 × 384 = 147,456 | 1× (baseline) |
| LoRA r=4 | 4×384 + 384×4 = 3,072 | 48× reduction |
| LoRA r=8 | 8×384 + 384×8 = 6,144 | 24× reduction |
| LoRA r=16 | 16×384 + 384×16 = 12,288 | 12× reduction |

### Why inject into Q, K, V specifically?

The attention mechanism is where the model learns *which residues to attend to*. For a new task (e.g., mutation prediction), we want to change the model's "attention patterns" — which pairs of residues it focuses on. The value projection V carries the actual information; Q and K determine relevance. Injecting LoRA into all three gives maximal task adaptation with minimal parameters.

### 📖 Reading Guide — LoRA — Low-Rank Adaptation

`class LoRALinear(nn.Module):`
→ *A replacement for a standard nn.Linear layer that adds two small 'adapter' matrices instead of modifying the original weights.*

`self.weight = nn.Parameter(pretrained_weight, requires_grad=False)`
→ *The original pretrained weight is FROZEN — requires_grad=False means it will NOT be updated during training.*

`self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)`
→ *First adapter matrix: shape (rank × in_features). rank is tiny (e.g. 8) compared to in_features (e.g. 256).*

`self.lora_B = nn.Parameter(torch.zeros(out_features, rank))`
→ *Second adapter matrix: initialised to zero so LoRA starts as a no-op (doesn't change the model at all initially).*

`def forward(self, x):`
→ *Called on every forward pass. x is the input tensor (batch of embeddings).*

`base_out = F.linear(x, self.weight)`
→ *Standard matrix multiply with the frozen original weights. This is what the pretrained model would have done.*

`lora_out = x @ self.lora_A.T @ self.lora_B.T`
→ *The adaptation: x → small space (rank) → back to full space. Only 2×rank×d parameters instead of d².*

`return base_out + self.scaling * lora_out`
→ *Add the small correction on top of the frozen output. scaling=alpha/rank controls how much LoRA contributes.*



In [ ]:
# Section 3: LoRA — Low-Rank Adaptation
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class LoRALayer(nn.Module):
    """LoRA: W_effective = W_frozen + alpha/r * B @ A
    where W is (d_out, d_in), A is (r, d_in), B is (d_out, r).
    """
    def __init__(self, d_in, d_out, rank=4, alpha=16, dropout=0.0):
        """Initialise a LoRA adapter layer.

Adds trainable low-rank matrices A and B to a frozen linear projection.
Effective output: W·x + (B·A·x)·(alpha/rank).

Args:
    d_in (int): input feature dimension.
    d_out (int): output feature dimension.
    rank (int): LoRA rank r — controls adapter parameter count.
    alpha (float): scaling factor; effective scale = alpha/rank.
    dropout (float): dropout probability applied to A·x."""
        super().__init__()
        self.d_in = d_in; self.d_out = d_out; self.rank = rank
        self.scaling = alpha / rank

        # Frozen pretrained weights
        self.weight = nn.Parameter(
            nn.init.kaiming_uniform_(torch.empty(d_out, d_in), a=math.sqrt(5)),
            requires_grad=False
        )
        self.bias = nn.Parameter(torch.zeros(d_out), requires_grad=False)

        # LoRA matrices (trainable)
        self.lora_A = nn.Parameter(torch.randn(rank, d_in) * 0.02)  # Gaussian init
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))         # Zero init → ΔW=0 at start
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """Apply the frozen linear weight plus the LoRA low-rank correction.

Args:
    x (Tensor): input tensor of shape (..., d_in).

Returns:
    Tensor: output of shape (..., d_out)."""
        base = F.linear(x, self.weight, self.bias)
        delta = F.linear(self.dropout(x), self.lora_A)  # (*, rank)
        delta = F.linear(delta, self.lora_B)             # (*, d_out)
        return base + self.scaling * delta

    def merge_weights(self):
        """Merge LoRA into W for inference speedup."""
        merged_W = self.weight + self.scaling * (self.lora_B @ self.lora_A)
        return merged_W

# Demonstration
d_in, d_out, rank = 256, 256, 8
lora = LoRALayer(d_in, d_out, rank=rank, alpha=16)
full_linear = nn.Linear(d_in, d_out)

trainable_lora = sum(p.numel() for p in lora.parameters() if p.requires_grad)
total_lora = sum(p.numel() for p in lora.parameters())
print(f"d_model={d_in}, rank={rank}")
print(f"Total params: {total_lora:,}")
print(f"Trainable (LoRA only): {trainable_lora:,} ({trainable_lora/total_lora:.1%})")
print(f"Full linear: {sum(p.numel() for p in full_linear.parameters()):,}")

x = torch.randn(4, 32, d_in)
out = lora(x)
print(f"\nForward: {x.shape} → {out.shape}")
print(f"ΔW at init (should be ~0): {(lora.lora_B @ lora.lora_A).abs().max():.6f}")

### 📖 Reading Guide — ΔΔG Prediction Head

`class DeltaDeltaGHead(nn.Module):`
→ *A regression head on top of the structure model. Outputs a single number (ΔΔG) per mutation.*

`self.pair_pool = nn.Linear(c_z, hidden)`
→ *c_z = pair representation dimension (128 in AF3). Pool the pair features around the mutation site.*

`ddg = self.output(hidden_repr)`
→ *Final linear layer: maps to 1 number. Negative = stabilising. Positive = destabilising.*

`loss = F.mse_loss(ddg.squeeze(), target_ddg)`
→ *Mean squared error between predicted and experimental ΔΔG. Training signal.*



## Section 4: The Task — Mutation Effect Prediction (ΔΔG)

### What is ΔΔG?

ΔΔG = G(mutant) − G(wildtype): the change in folding free energy caused by a point mutation.

- **ΔΔG > 0:** Mutation **destabilizes** the protein (weaker folding). Relevant for understanding disease variants.
- **ΔΔG < 0:** Mutation **stabilizes** the protein. Useful in enzyme engineering and therapeutic protein design.
- **ΔΔG ≈ 0:** Neutral mutation. No significant effect on stability.

### Dataset: FireProtDB

- ~4,000 experimentally measured single-point mutations with ΔΔG values
- Covers 176 protein families
- Values range roughly −5 to +5 kcal/mol (most are near 0)
- URL: https://loschmidt.chemi.muni.cz/fireprotdb/

### Model Input/Output

```
Input:
  - tokens_wt:  (B, N) integer sequence (wildtype)
  - tokens_mut: (B, N) integer sequence (mutant, differs at position mut_pos)
  - mut_pos:    (B,)   position of the mutation

Output:
  - ΔΔG:        (B,)   predicted change in folding free energy (kcal/mol)
```

### Why a Siamese / Difference Architecture?

We run the Pairformer on **both** wildtype and mutant sequences, then subtract:
- Δs = s_mut − s_wt (difference in single representation)
- Δz = z_mut − z_wt (difference in pair representation)

This is equivariant to global sequence properties (things unchanged by mutation cancel out). Only mutation-induced changes survive. The head then pools the local neighborhood around the mutation site.

### Evaluation Metrics

- **Pearson r:** Linear correlation between predicted and experimental ΔΔG. State-of-the-art: ~0.7
- **Spearman ρ:** Rank correlation — more robust to outliers. Preferred in practice.
- **MAE:** Mean absolute error in kcal/mol. Useful for engineering applications.

In [ ]:
# Section 4: ΔΔG mutation effect prediction
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class DDGPredictor(nn.Module):
    """Predict ΔΔG (stability change upon mutation) from Pairformer embeddings.
    ΔΔG = ΔG(mutant) - ΔG(wild-type): negative = stabilizing, positive = destabilizing.
    """
    def __init__(self, c_s=128, c_z=64):
        """Initialise the ΔΔG prediction head.

Args:
    c_s (int): single representation dimension from Pairformer.
    c_z (int): pair representation dimension from Pairformer."""
        super().__init__()
        # Mutation embedding: encode AA change
        self.aa_embed = nn.Embedding(21, 32)  # 20 AAs + gap
        # Structure context from Pairformer single repr
        self.struct_proj = nn.Linear(c_s, 64)
        # Pair context at mutation site
        self.pair_proj = nn.Linear(c_z, 32)
        # Regression head
        self.head = nn.Sequential(
            nn.Linear(32 + 64 + 32, 128), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, wt_aa, mut_aa, position, s, z):
        """
        wt_aa, mut_aa: (B,) wild-type and mutant AA indices
        position: (B,) mutation position
        s: (B, L, c_s) single repr from Pairformer
        z: (B, L, L, c_z) pair repr
        """
        aa_feat = self.aa_embed(mut_aa) - self.aa_embed(wt_aa)  # (B, 32)
        # Extract structure context at mutation site
        struct_feat = self.struct_proj(s[torch.arange(s.shape[0]), position])  # (B, 64)
        # Extract pair context at mutation site (mean of row)
        pair_row = z[torch.arange(z.shape[0]), position].mean(dim=1)  # (B, c_z)
        pair_feat = self.pair_proj(pair_row)  # (B, 32)

        x = torch.cat([aa_feat, struct_feat, pair_feat], dim=-1)
        return self.head(x).squeeze(-1)

torch.manual_seed(42)
B, L, c_s, c_z = 8, 50, 128, 64
model = DDGPredictor(c_s=c_s, c_z=c_z)
wt_aa = torch.randint(0, 20, (B,))
mut_aa = torch.randint(0, 20, (B,))
position = torch.randint(0, L, (B,))
s = torch.randn(B, L, c_s); z = torch.randn(B, L, L, c_z)

ddg_pred = model(wt_aa, mut_aa, position, s, z)
print(f"ΔΔG predictions (kcal/mol): {ddg_pred.detach().numpy().round(2)}")
print(f"<0 = stabilizing, >0 = destabilizing")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Section 5: Three Fine-Tune Modes Compared

| Mode | Pairformer | Embedder + Head | Use when |
|---|---|---|---|
| **frozen** | 0 params trainable | All trainable | <500 examples, fast baseline experiments |
| **lora** | LoRA A+B only (~0.5%) | All trainable | 500–10K examples, standard approach |
| **full** | All params trainable | All trainable | >10K examples, large GPU budget |

**When to use each:**

- **Frozen** is the fastest sanity check. If frozen achieves reasonable correlation, the pretrained representations already capture the biology.
- **LoRA** is the default for protein ML. You get task-specific adaptation without risking catastrophic forgetting of geometric structure.
- **Full fine-tuning** risks overfitting on small datasets and can destroy the pretrained geometric representations. Only justified with large, high-quality experimental datasets.

In [ ]:
# Section 5: Three fine-tune modes compared
import torch
import torch.nn as nn
import time

class ToyPairformer(nn.Module):
"""Minimal Pairformer for demonstration and finetuning experiments.

Stacks n_blocks of triangle self-attention on the pair representation
and a 2-layer transition on the single representation.

Args:
    c_z (int): pair representation channel dimension (default 128).
    c_s (int): single representation channel dimension (default 256).
    n_blocks (int): number of stacked Pairformer blocks (default 4)."""
    def __init__(self, c_z=128, c_s=256, n_blocks=4):
        """Build projection layers and stack n_blocks attention+transition blocks.

Args:
    c_z (int): pair representation channel dimension.
    c_s (int): single representation channel dimension.
    n_blocks (int): depth of the Pairformer stack."""
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(c_z), nn.Linear(c_z, c_z*2), nn.GELU(), nn.Linear(c_z*2, c_z)
            ) for _ in range(n_blocks)
        ])
        self.single_net = nn.Linear(c_s, c_s)
    def forward(self, z, s):
        """Run one forward pass through all Pairformer blocks.

Args:
    z (Tensor): pair representation of shape (B, L, L, c_z).
    s (Tensor): single representation of shape (B, L, c_s).

Returns:
    Tuple[Tensor, Tensor]: updated (z, s) tensors."""
        for block in self.blocks:
            z = z + block(z)
        return z, s + self.single_net(s)

# Mode 1: Head-only
def head_only_mode(model):
"""Freeze all Pairformer parameters; only the ΔΔG prediction head trains.

This is the fastest finetuning strategy but gives the lowest accuracy
when the new task distribution differs significantly from pretraining.

Args:
    model: ToyPairformer instance whose backbone parameters are frozen."""
    for p in model.parameters():
        p.requires_grad = False
    return model

# Mode 2: Full fine-tuning
def full_finetune_mode(model):
"""Unfreeze all model parameters for end-to-end finetuning.

Gives the highest accuracy but risks catastrophic forgetting of
pretraining knowledge and requires more labelled data to converge.

Args:
    model: ToyPairformer instance — all parameters set requires_grad=True."""
    for p in model.parameters():
        p.requires_grad = True
    return model

# Mode 3: LoRA injection (simplified)
def lora_mode(model, rank=4):
"""Replace all nn.Linear layers with LoRA adapters; freeze base weights.

Only the low-rank A and B matrices (and bias) are trained, drastically
reducing the number of trainable parameters while preserving capacity.

Args:
    model: ToyPairformer instance to inject LoRA adapters into.
    rank (int): LoRA rank r (default 4). Higher rank = more parameters."""
    for p in model.parameters():
        p.requires_grad = False
    # Add trainable LoRA params to last block only
    c_z = 128
    model.lora_A = nn.Parameter(torch.randn(rank, c_z) * 0.01)
    model.lora_B = nn.Parameter(torch.zeros(c_z, rank))
    return model

torch.manual_seed(42)
for mode_name, mode_fn in [('Head-only', head_only_mode),
                             ('Full fine-tune', full_finetune_mode),
                             ('LoRA', lora_mode)]:
    m = ToyPairformer(c_z=128, c_s=256, n_blocks=4)
    m = mode_fn(m)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    print(f"{mode_name:20s}: {trainable:8,} / {total:8,} trainable ({trainable/total:.1%})")

## Section 6: LR Schedule — Warmup + Cosine Decay

### Why Warmup Is Critical for Transformer Fine-Tuning

At the start of fine-tuning, the task head is randomly initialized. Its gradients are large and noisy. If the learning rate is immediately at its peak value, these large gradients will propagate back into the Pairformer and corrupt the carefully learned geometric representations in the first few steps — potentially unrecoverable.

**Warmup:** LR increases linearly from 0 → max_lr over the first ~500 steps. This gives the randomly initialized head time to stabilize before the Pairformer starts receiving meaningful gradient signal.

**Cosine decay:** After warmup, LR anneals smoothly from max_lr → min_lr. Cosine is preferred over linear decay because it keeps LR high during productive middle training, then ramps down as the loss landscape flattens near convergence.

### Differential Learning Rate

Give the Pairformer a **10× lower learning rate** than the task head:

```python
optimizer = AdamW([
    {'params': pairformer_params, 'lr': max_lr * 0.1},  # preserve learned structure
    {'params': other_params,      'lr': max_lr},          # fast adaptation for head
])
```

This preserves the learned geometric representations while allowing the head to adapt quickly. It is especially important with LoRA: even the small LoRA A/B updates should be conservative to avoid overfitting.

### Mixed Precision: bfloat16 vs float16

- **float16:** 5-bit exponent, 10-bit mantissa. Range: ≈ ±65,504. Can overflow during forward pass for protein models with long sequences.
- **bfloat16:** 8-bit exponent, 7-bit mantissa. Same range as float32 (≈ ±3.4 × 10³⁸). Never overflows; lower precision is acceptable for activations.
- **Protein LMs use bfloat16** because the pair representation z can have large values after triangle attention, and float16 overflow would corrupt training silently.

In [ ]:
# Section 6: LR schedule — warmup + cosine decay
import torch
import numpy as np
import matplotlib.pyplot as plt

def warmup_cosine_schedule(step, warmup_steps=500, total_steps=5000, min_lr=1e-6, max_lr=1e-4):
    """Learning rate schedule used in large protein model fine-tuning."""
    if step < warmup_steps:
        # Linear warmup
        return max_lr * step / warmup_steps
    else:
        # Cosine decay to min_lr
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        cosine = 0.5 * (1 + np.cos(np.pi * progress))
        return min_lr + (max_lr - min_lr) * cosine

steps = np.arange(0, 5000)
lrs = [warmup_cosine_schedule(s) for s in steps]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(steps, lrs)
ax.axvline(500, color='r', linestyle='--', label='End of warmup')
ax.set_xlabel('Step'); ax.set_ylabel('Learning rate')
ax.set_title('Warmup + Cosine Decay LR Schedule')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/lr_schedule.png', dpi=72)
print(f"Peak LR: {max(lrs):.2e}")
print(f"Final LR: {lrs[-1]:.2e}")
print(f"Warmup ratio: {500/5000:.0%}")
print("Why warmup: random init → large gradients early → warmup prevents divergence")

## Section 7: LoRA Checkpointing — Save Adapters, Not Full Models

### The Size Problem

A full Boltz-2 checkpoint is approximately 2 GB (1B parameters × 2 bytes/param in bfloat16). Saving one checkpoint per fine-tuning run is expensive. Serving multiple task-specific models from a single server is infeasible.

### The LoRA Solution

Save **only the trainable parameters** (LoRA A, B matrices + embedder + head). For a rank-8 adapter on a standard Pairformer:
- Full model: ~2 GB
- LoRA adapter: ~10 MB (200× smaller)

**Inference workflow:**
1. Load frozen base model once (shared across all tasks)
2. Load task-specific LoRA adapter (~10 MB per task)
3. Inject adapter into base model → run inference

**Merge workflow (zero inference overhead):**
- W_merged = W + (α/r) × B @ A
- Replace all LoRALinear with plain nn.Linear using merged weights
- No runtime overhead at all — identical to the original model architecture

In [ ]:
# Section 7: LoRA checkpointing
import torch
import torch.nn as nn
import os

def save_lora_checkpoint(model, path, metadata=None):
    """Save only LoRA weights (tiny checkpoint)."""
    lora_state = {k: v for k, v in model.state_dict().items()
                  if 'lora_' in k}
    checkpoint = {
        'lora_state_dict': lora_state,
        'metadata': metadata or {},
        'n_params': len(lora_state),
    }
    torch.save(checkpoint, path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"Saved LoRA checkpoint: {len(lora_state)} tensors, {size_mb:.3f} MB")
    return size_mb

def load_lora_checkpoint(model, path):
    """Load LoRA weights into model."""
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['lora_state_dict'], strict=False)
    print(f"Loaded LoRA checkpoint: {ckpt['metadata']}")
    return model

# Demo
torch.manual_seed(42)
class LoRAModel(nn.Module):
"""Toy model demonstrating LoRA checkpointing.

Contains a single LoRA-adapted linear layer to illustrate how adapter
weights are saved and reloaded independently of the frozen base model."""
    def __init__(self):
        """Initialise with a single LoRA-adapted linear layer (in=10, out=5, rank=4)."""
        super().__init__()
        self.backbone = nn.Linear(128, 128)  # frozen
        self.backbone.weight.requires_grad = False
        self.lora_A = nn.Parameter(torch.randn(8, 128) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(128, 8))
        self.head = nn.Linear(128, 1)
    def forward(self, x):
        """Apply the frozen linear weight plus the LoRA low-rank correction.

Args:
    x (Tensor): input tensor of shape (..., d_in).

Returns:
    Tensor: output of shape (..., d_out)."""
        return self.head(x + (x @ self.lora_A.T) @ self.lora_B.T)

model = LoRAModel()
total_mb = sum(p.numel()*4 for p in model.parameters()) / 1e6
trainable_mb = sum(p.numel()*4 for p in model.parameters() if p.requires_grad) / 1e6
print(f"Full model: {total_mb:.3f} MB")
print(f"Trainable (LoRA + head): {trainable_mb:.3f} MB")

save_lora_checkpoint(model, '/tmp/lora_checkpoint.pt',
                     metadata={'task': 'ddg', 'rank': 8, 'steps': 1000})

## Section 8: Scaling to Real Protein Structure Models

The toy model in this notebook uses tiny dimensions (c_s=64, c_z=32, 2 blocks) to run quickly on CPU. The same code architecture, with larger dimensions, becomes Boltz-2 / AlphaFold3 scale.

### Toy vs. Production Dimensions

```
                    Toy (this notebook)    AF3/Boltz-2 scale
c_s (single)              64                   384
c_z (pair)                32                   128
n_blocks                   2                    48
n_heads                    4                    16
N (sequence length)       48                   384
Memory per forward    ~20 MB                ~4 GB
LoRA trainable (r=8)    ~5K params         ~300K params
Full model            ~500K params          ~1B params
```

### Boltz-2 Specifically

**Boltz-2** (github.com/jwohlwend/boltz) is an open-source reproduction of AlphaFold3 mechanics released under the MIT license — meaning you can legally fine-tune it.

Key features:
- **Inputs:** Protein chains + DNA/RNA chains + small molecule ligands (CCD format)
- **Output:** 3D atomic coordinates + confidence scores (pLDDT per residue, PAE between residue pairs, ipTM for complex quality)
- **Architecture:** Full AF3-style: MSA module → Pairformer → Diffusion-based coordinate generation
- **LoRA target:** Pairformer Q/K/V and triangle operation projections
- **Fine-tuning recipe:** r=16, α=32, warmup 500 steps, max_lr=1e-4, differential LR 10x

In [ ]:
# Section 8: Scaling to real protein structure models
import torch
import torch.nn as nn

print("Scaling LoRA to Real Protein Structure Models")
print("=" * 60)
print()

real_models = {
    "ESM-2 (650M)": {
        "params": "650M", "d_model": 1280, "n_layers": 33,
        "lora_rank": "8-16", "trainable_pct": "~0.1%",
        "gpu": "1× A100 (40GB)", "task": "Sequence → function"
    },
    "ESMFold": {
        "params": "690M", "d_model": 1024, "n_layers": 48,
        "lora_rank": "8", "trainable_pct": "~0.05%",
        "gpu": "2× A100", "task": "Sequence → structure (single chain)"
    },
    "AlphaFold2": {
        "params": "93M", "d_model": 256, "n_layers": 48,
        "lora_rank": "4-8", "trainable_pct": "~2%",
        "gpu": "1× A100", "task": "MSA + template → structure"
    },
    "AlphaFold3": {
        "params": "~300M", "d_model": 384, "n_layers": 48,
        "lora_rank": "8-16", "trainable_pct": "~1%",
        "gpu": "2-4× A100", "task": "All-atom complex structure"
    },
    "Boltz-2": {
        "params": "~200M", "d_model": 256, "n_layers": 24,
        "lora_rank": "8", "trainable_pct": "~2%",
        "gpu": "1× A100", "task": "Open-source AF3 alternative"
    },
}

header = f"{'Model':<20} {'Params':<12} {'LoRA rank':<12} {'Trainable%':<12} {'GPU':<20}"
print(header)
print("-" * len(header))
for name, info in real_models.items():
    print(f"{name:<20} {info['params']:<12} {info['lora_rank']:<12} {info['trainable_pct']:<12} {info['gpu']:<20}")

print()
print("Key principle: LoRA rank r controls the bottleneck dimension.")
print("r=4: minimal adaptation (~99.9% frozen)")
print("r=16: more capacity (~99.6% frozen)")
print("r=64: approaches full fine-tuning")

## Section 9: TCR-pMHC Capstone Project

### The Biology

T-cell receptors (TCRs) recognize foreign peptides displayed by MHC molecules on cell surfaces. This recognition triggers the adaptive immune response. The specificity is determined by the CDR3β loop — a hypervariable ~15-residue segment.

**Clinical relevance:**
- Cancer immunotherapy: designing TCRs that target tumor neoantigens
- Autoimmune disease: understanding which self-peptides are accidentally recognized
- Vaccine design: predicting which peptides will be immunogenic

### The Prediction Challenge

Current best tool: **TCRmodel2** (Yin et al. 2023) — CDR3β RMSD ≈ 2.5 Å on test structures.

**Our goal:** Fine-tune Boltz-2 on STCRDab → CDR3β RMSD < 1.5 Å.

### The Pipeline

```
STCRDab structures (500+ TCR-pMHC complexes)
    ↓ extract CDR3β sequences + Cα coordinates
    ↓ sequence identity split (<30% identity: train vs. test)
Boltz-2 base model (frozen, 1B params)
    + LoRA r=16 on Pairformer Q/K/V
    + CDR3-specific position embeddings (mark CDR3 residues in input)
    ↓ fine-tune 10K steps on 400 training complexes
    ↓ evaluate on 30 held-out structures
Target: CDR3β RMSD < 1.5 Å (beat TCRmodel2 by 40%)
```

### Why Sequence Identity Splitting Matters

Proteins share evolutionary history. If you split randomly, similar sequences end up in both train and test — the model memorizes, not generalizes. Splitting at <30% sequence identity ensures the model must generalize to genuinely novel CDR3β sequences.

In [ ]:
# Section 9: TCR-pMHC capstone project
import torch
import torch.nn as nn
import numpy as np

print("TCR-pMHC Structure Prediction Capstone")
print("=" * 60)
print()
print("Biology:")
print("  TCR (T-cell receptor) recognizes peptide-MHC (pMHC) complexes")
print("  CDR3β loop makes key contacts with peptide")
print("  Goal: predict CDR3β RMSD < 1.5 Å (clinically relevant)")
print()
print("Data: STCRDab (Structural T-Cell Receptor Database)")
print("  ~3000 TCR-pMHC structures from PDB")
print("  Input: TCR sequence + pMHC sequence")
print("  Output: 3D coordinates of CDR3β loop (avg 12 residues)")
print()

# Simulate CDR3β prediction evaluation
np.random.seed(42)
n_test = 100
true_cdr3 = np.random.randn(n_test, 12, 3) * 5  # 12-residue CDR3β

# Good model: Boltz-2 + LoRA fine-tuning
pred_cdr3_good = true_cdr3 + np.random.randn(n_test, 12, 3) * 0.8

# Baseline model: no fine-tuning
pred_cdr3_base = true_cdr3 + np.random.randn(n_test, 12, 3) * 2.5

def compute_rmsd(pred, true):
"""Compute root-mean-square deviation between predicted and true coordinates.

Args:
    pred (Tensor): predicted coordinates of shape (N, 3).
    true (Tensor): ground-truth coordinates of shape (N, 3).

Returns:
    float: RMSD in the same units as the input coordinates."""
    diff = pred - true
    return np.sqrt((diff**2).sum(axis=(1,2)) / (pred.shape[1]*3))

def kabsch_rmsd_batch(pred, true):
    """Simplified RMSD after centering (no rotation for speed)."""
    pred_c = pred - pred.mean(axis=1, keepdims=True)
    true_c = true - true.mean(axis=1, keepdims=True)
    return np.sqrt(((pred_c - true_c)**2).sum(axis=(1,2)) / pred.shape[1])

rmsd_good = kabsch_rmsd_batch(pred_cdr3_good, true_cdr3)
rmsd_base = kabsch_rmsd_batch(pred_cdr3_base, true_cdr3)

print(f"Fine-tuned model:")
print(f"  Median CDR3β RMSD: {np.median(rmsd_good):.2f} Å")
print(f"  % < 1.5 Å: {(rmsd_good < 1.5).mean():.1%}")
print(f"\nBaseline (no fine-tuning):")
print(f"  Median CDR3β RMSD: {np.median(rmsd_base):.2f} Å")
print(f"  % < 1.5 Å: {(rmsd_base < 1.5).mean():.1%}")
print(f"\nTarget: median CDR3β RMSD < 1.5 Å ✓" if np.median(rmsd_good) < 1.5 else "Target not met")

## Interview Q&A — Fine-Tuning Protein Structure Models

These questions and answers reflect what interviewers at Genentech, drug discovery companies, computational biology ML teams, and similar ML-driven biotech companies ask in ML scientist interviews.

---

**Q1: Why LoRA rather than full fine-tuning for protein LMs with small datasets?**

A: Three reasons. (1) **Catastrophic forgetting:** full fine-tuning on 500 examples will overwrite the geometric representations (triangle inequalities, co-evolution patterns) that took AF3 months to learn on PDB. LoRA's frozen backbone preserves them. (2) **Overfitting:** fewer trainable parameters = stronger implicit regularization on small datasets. (3) **Practical:** a 10 MB adapter vs. a 2 GB checkpoint is easier to version, share, and deploy in a multi-task serving system.

---

**Q2: Why initialize LoRA matrix B to zero?**

A: To guarantee that at training step 0, the fine-tuned model is **identical** to the pretrained model. ΔW = B @ A. If B=0 then ΔW=0 regardless of A. This means the first gradient step is taken from a meaningful starting point (the pretrained optimum) rather than from a random perturbation. If both A and B were randomly initialized, the model would immediately diverge from the pretrained state before it has seen any task data.

---

**Q3: Why use LR warmup for transformer fine-tuning?**

A: At the start of fine-tuning, the task head is randomly initialized and produces large, noisy gradients. If the LR is immediately at peak, these gradients propagate back through the Pairformer with high magnitude, potentially corrupting learned attention patterns in the first few steps. Warmup gives the head time to stabilize before the backbone receives meaningful signal. For LoRA specifically: even the small A/B matrices need a few steps to orient toward the task before large updates.

---

**Q4: Why use differential LR (10x lower for Pairformer)?**

A: The Pairformer already knows protein geometry; we want to preserve that while the task head learns to read out ΔΔG. Using the same LR for both would either over-train the Pairformer (corrupting geometry) or under-train the head (slow convergence). A 10x ratio is a common heuristic: aggressive enough to adapt LoRA adapters, conservative enough to preserve backbone representations. In practice you tune this ratio on a validation set.

---

**Q5: bfloat16 vs float16 — which do you use for protein LMs and why?**

A: bfloat16. The critical difference is the exponent range: float16 has a max value of ~65,504 and will overflow for large activations (which occur in pair representations after triangle attention on long sequences). bfloat16 uses 8 exponent bits (same as float32), giving max ~3.4×10³⁸ — it never overflows. The reduced mantissa precision (7 bits vs. 10 bits for float16) is acceptable for activations. For gradients and optimizer states, float32 is still used via the mixed-precision scaler.

---

**Q6: How do you prevent catastrophic forgetting during fine-tuning?**

A: Multiple complementary approaches: (1) **LoRA** — most effective; frozen backbone cannot forget what it doesn't update. (2) **Low LR on backbone** — differential LR limits how much the backbone can change per step. (3) **Early stopping on validation set** — stop before over-adaptation. (4) **Elastic Weight Consolidation (EWC)** — add regularization term penalizing deviation from pretrained weights, weighted by parameter importance (Fisher information). (5) **Data mixing** — include a fraction of generic protein data in fine-tuning to anchor representations.

---

**Q7: How do you evaluate ΔΔG prediction models?**

A: Three metrics, each capturing something different. (1) **Pearson r** — linear correlation; good for absolute ΔΔG accuracy. (2) **Spearman ρ** — rank correlation; more robust to outliers, preferred when you care about ranking mutations (e.g., which of 100 variants to synthesize). (3) **MAE** (mean absolute error in kcal/mol) — practical error for engineering applications; 1 kcal/mol is a meaningful threshold for stability engineering. Always report all three. Do NOT use RMSE alone — it penalizes outliers heavily and obscures whether the model is systematically biased.

---

**Q8: What is sequence identity splitting and why is it critical?**

A: Standard random train/test split assigns similar sequences to both sets. If proteins A and B share 80% sequence identity and A is in train, B's ΔΔG can essentially be looked up by the model (sequence similarity ≈ structural similarity). This inflates test metrics — the model appears to generalize but is actually interpolating. Sequence identity splitting at <30% threshold ensures test proteins are genuinely novel: no training protein shares >30% identity with any test protein. This is the standard for protein ML evaluation (used in CASP, ProteinGym, etc.). Without it, published results are not reproducible on truly unseen proteins.

In [ ]:
# Interview Q&A — Fine-tuning Protein Structure Models
print("=" * 60)
print("INTERVIEW Q&A — FINE-TUNING PROTEIN STRUCTURE MODELS")
print("=" * 60)

qas = [
    ("Q: Why is LoRA preferred over full fine-tuning for protein models?",
     "A: Protein structure models (AF3, ESM2) have 93M-690M parameters. Full fine-tuning "
     "requires storing gradients for all weights — infeasible on single GPUs for most labs. "
     "LoRA trains only rank-r decomposition matrices (~0.1-2% of params), achieving similar "
     "performance with 50-100× less GPU memory. Also: smaller checkpoints, no catastrophic "
     "forgetting of pretrained structure knowledge."),

    ("Q: What is ΔΔG and why does it matter?",
     "A: ΔΔG = ΔG(mutant) - ΔG(wildtype). Negative = mutation stabilizes the protein "
     "(lower folding energy). Positive = destabilizing. Critical for: drug resistance mutations, "
     "thermostable enzyme engineering, antibody optimization. Accurate ΔΔG prediction from "
     "sequence+structure is an open problem; current models (Rosetta, ThermoMPNN, ESM-IF) "
     "achieve Pearson r ~ 0.6-0.7 on held-out mutations."),

    ("Q: What evaluation metric do you use for CDR3β loop prediction?",
     "A: RMSD after Kabsch alignment (optimal rotation). For CDR3β loops specifically: "
     "RMSD < 1.5Å = good (loop is essentially correct), 1.5-3Å = acceptable, >3Å = incorrect. "
     "TM-score is not appropriate for short loops. lDDT-Cα is also useful for local accuracy."),

    ("Q: What is catastrophic forgetting and how does LoRA prevent it?",
     "A: When fine-tuning on a new task, the model overwrites old weights, 'forgetting' "
     "the pretraining knowledge. LoRA prevents this by keeping W frozen and only learning "
     "ΔW = B@A in a separate low-rank subspace. The frozen W retains all structure knowledge; "
     "ΔW captures only the task-specific adaptation."),

    ("Q: What learning rate do you use for Pairformer fine-tuning?",
     "A: Much lower than pretraining. Typical: 1e-5 to 5e-5 for LoRA params, with "
     "warmup (500-1000 steps) + cosine decay. High LR destroys pretrained representations. "
     "Rule of thumb: 10-100× lower than original pretraining LR."),
]

for q, a in qas:
    print(f"\n{q}")
    print(f"  {a}")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **ΔΔG thermodynamics**: ΔΔG = ΔG_mut − ΔG_wt = −RT ln(K_mut/K_wt) — free energy perturbation; negative = stabilizing mutation
- **LoRA intrinsic dimensionality**: hypothesis that pre-trained models have low intrinsic dimensionality — rank r≪d suffices for fine-tuning
- **LoRA math**: W' = W + α·(B·A), A ∈ ℝ^(r×d_in), B ∈ ℝ^(d_out×r), trainable params = 2·r·d (vs d² for full fine-tuning)
- **Transfer learning theory**: domain shift (covariate shift, label shift), catastrophic forgetting — LoRA mitigates both via frozen backbone
- **TCR-pMHC immunology**: T-cell receptor binds peptide-major histocompatibility complex; structural prediction enables neoantigen vaccine design

### 2️⃣ Must-Have Popular Resources
#### 🟢 Start Here (Before Fine-tuning)
- 🆓 **HuggingFace PEFT Tutorial** — [huggingface.co/docs/peft/tutorial/peft_model_train](https://huggingface.co/docs/peft/tutorial/peft_model_train) — official LoRA guide
- 🆓 **LoRA Paper (Hu 2021)** — [arxiv.org/abs/2106.09685](https://arxiv.org/abs/2106.09685) — read it, only 12 pages
- 🆓 **ESM2 HuggingFace** — [huggingface.co/facebook/esm2_t6_8M_UR50D](https://huggingface.co/facebook/esm2_t6_8M_UR50D) — start with 8M model, then scale up
- 🆓 **ProteinGym GitHub** — [github.com/OATML-Markslab/ProteinGym](https://github.com/OATML-Markslab/ProteinGym) — the benchmark you'll evaluate on
- 📘 LoRA (Hu et al. 2021, arXiv 2106.09685) — 4-page essential read; introduces rank decomposition for LLMs
- 📘 ESM-1v (Meier 2021) — zero-shot mutation effect prediction with protein language models
- 📘 ProteinGym benchmark (Notin 2023) — 250 deep mutational scanning datasets; standard evaluation for mutation prediction
- 📘 Boltz-2 technical report — most recent AF3-alternative with fine-tuning support
- ⭐ GitHub [huggingface/peft](https://github.com/huggingface/peft) 16k★ — LoRA, QLoRA, IA³, prefix tuning
- ⭐ GitHub [OATML-Markslab/ProteinGym](https://github.com/OATML-Markslab/ProteinGym) 5k★ — benchmark + leaderboard

### 3️⃣ Practicals — Hands-On
- 💻 Fine-tune ESM2 with LoRA (rank=8, alpha=16) on ProteinGym thermostability DMS — target Pearson r > 0.5
- 💻 Compare frozen / LoRA / full fine-tuning modes: log loss, validation Pearson, parameter count, runtime
- 💻 Evaluate on held-out mutations: compute Spearman correlation (ProteinGym standard metric)
- 🤗 HuggingFace: [EvolutionaryScale/esm3-sm-open-v1](https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1) — ESM3 multimodal model
- 🤗 HuggingFace: [OATML-Markslab/ProteinGym](https://huggingface.co/datasets/OATML-Markslab/ProteinGym) — 250 DMS datasets
- 📊 Kaggle: [Novozymes Enzyme Stability Prediction](https://www.kaggle.com/competitions/novozymes-enzyme-stability-prediction) — perfect ΔΔG benchmark

### 4️⃣ Real-World Problems
- 🏭 computational biology ML teams: AF3 fine-tuning for drug lead optimization — custom LoRA adapters per target class
- 🏭 Absci: antibody design using fine-tuned protein language models — LoRA enables target-specific CDR prediction
- 🏭 drug discovery companies: de novo protein design with fine-tuned generative models
- 📊 Dataset: [ProteinGym 250 DMS](https://huggingface.co/datasets/OATML-Markslab/ProteinGym) — deep mutational scanning across diverse proteins
- 🔬 Application: TCR neoantigen prediction — fine-tune on pMHC binding data for personalized cancer vaccine design

### 5️⃣ Interview Tips
- ❓ Must know: LoRA parameter count — for a d×d layer with rank r: 2·r·d trainable params; e.g. d=1280, r=8 → 20,480 vs 1,638,400 (full)
- ❓ Must know: Why differential learning rate? — pre-trained layers encode general protein knowledge; lower LR preserves it; new head needs higher LR
- ❓ Must know: LoRA merge for zero inference overhead — `W_merged = W + α·B·A`; replace module before serving
- ❓ Must know: ProteinGym metric — Spearman correlation between predicted and experimental ΔΔG across all single mutations in a DMS assay
- ⚠️ Common mistake: Using Pearson instead of Spearman for ΔΔG — experimental assays have outliers; Spearman is rank-based and more robust
- 💡 Pro tip: Always cite ProteinGym Spearman as your evaluation metric; interviewers at protein ML companies know this benchmark

### 6️⃣ Hot Industry Topics
- 🔥 Trending: [EvolutionaryScale/esm3](https://github.com/evolutionaryscale/esm3) — ESM3 multimodal protein LM (sequence + structure + function tokens)
- 🔥 Trending: [jwohlwend/boltz](https://github.com/jwohlwend/boltz) — Boltz-2, fine-tunable open-source AF3 alternative
- 🔥 Trending: [RosettaCommons/RFdiffusion](https://github.com/RosettaCommons/RFdiffusion) — diffusion-based de novo protein design
- 🚀 Build this: Fine-tune ESM2-650M with LoRA on the Novozymes Kaggle dataset, achieve Spearman r > 0.4 on the test set, and merge adapters for inference

### Time Complexity — Fine-tuning Protein Structure Models
| Operation | Time | Space | Notes |
|-----------|------|-------|-------|
| LoRA forward (r, d_in, d_out) | O(r·(d_in+d_out)) | O(2·r·d_in) | r << d |
| Full fine-tune (d × d) | O(d²) | O(d²) | all weights |
| Pairformer block (n residues) | O(n² × d) | O(n²) | — |
| Triangle attn (n residues) | O(n² × n × d_z) | O(n²) | n³ operations |
| Gradient checkpointing | O(forward²) | O(√n layers) | recompute activations |
| Mixed precision (bfloat16) | 2× faster | 0.5× memory | A100/H100 |
| Differential LR (Pairformer 10×) | same | same | just optimizer LR dict |
| LoRA checkpoint (r=8, 12 layers) | O(1) save | ~1 MB | vs ~2GB full model |
| TCR-pMHC RMSD eval (n pairs) | O(n × L²) | O(n) | per-complex RMSD |

In [ ]:
# Resources for 10/01
print("KEY RESOURCES — Module 10: Fine-Tuning Protein Structure Models")
print()
refs = {
    "Papers": [
        "Hu et al. 2022 — LoRA: Low-Rank Adaptation of Large Language Models (ICLR)",
        "Lin et al. 2023 — Evolutionary-scale prediction with ESM (Science)",
        "Tran et al. 2024 — ΔΔG prediction survey",
        "Fischer et al. 2022 — STCRDab: T-cell receptor structures",
    ],
    "Code": [
        "Hugging Face PEFT: github.com/huggingface/peft — LoRA library",
        "OpenFold: github.com/aqlaboratory/openfold — AF2 PyTorch reimplementation",
        "ESM: github.com/facebookresearch/esm",
        "Boltz: github.com/jwohlwend/boltz — open-source AF3",
    ],
    "Datasets": [
        "STCRDab: opig.stats.ox.ac.uk/webapps/stcrdab",
        "FireProtDB: ΔΔG stability mutations",
        "ThermoMutDB: thermostability mutations",
        "SKEMPI: protein-protein binding affinity mutations",
    ],
}
for cat, items in refs.items():
    print(f"\n{cat}:")
    for item in items:
        print(f"  * {item}")

## Real World Problems

### Problem 1 — ΔΔG Prediction on FireProtDB

**Dataset:** FireProtDB — ~4,000 single-point mutations with experimental ΔΔG values across 176 protein families.
- URL: https://loschmidt.chemi.muni.cz/fireprotdb/
- Download as CSV; columns: uniprot_id, position, wildtype_aa, mutant_aa, ddg_kcal_mol

**Task:** Fine-tune this notebook's MutationPredictor (or ESM2 backbone) with LoRA on FireProtDB. Split by protein family at <30% sequence identity. Target: Spearman ρ > 0.6.

**Starter code:**
```python
# 1. Download FireProtDB CSV
# 2. Fetch sequences from UniProt API
# 3. Tokenize using the 20-AA alphabet (ACDEFGHIKLMNPQRSTVWY + X=20)
# 4. Sequence identity split with mmseqs2 easy-cluster --min-seq-id 0.3
# 5. Run train_mutation_predictor with fine_tune_mode='lora'
```

---

### Problem 2 — Fine-tune ESM2 with LoRA (HuggingFace PEFT)

**Baseline:** ESM2 (evolutionary scale modeling) from Meta is a protein language model pretrained on 250M protein sequences. Available on HuggingFace: `facebook/esm2_t6_8M_UR50D` (8M params, runs on CPU).
- URL: https://huggingface.co/facebook/esm2_t6_8M_UR50D
- HuggingFace PEFT LoRA docs: https://huggingface.co/docs/peft/task_guides/token-cls

**Task:** Use HuggingFace PEFT to inject LoRA into ESM2's attention layers, fine-tune on ΔΔG, compare Spearman ρ vs. this notebook's from-scratch implementation.

```python
from transformers import EsmForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType

base = EsmForSequenceClassification.from_pretrained('facebook/esm2_t6_8M_UR50D', num_labels=1)
config = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
                    target_modules=['query', 'key', 'value'])
model = get_peft_model(base, config)
model.print_trainable_parameters()
```

---

### Problem 3 — Boltz-2 TCR-pMHC Fine-tuning (Capstone)

**Dataset:** STCRDab — Structural T-Cell Receptor Database, 500+ experimentally solved TCR-pMHC structures.
- URL: https://opig.stats.ox.ac.uk/webapps/stcrdab/
- Download: bulk PDB download via the "Download" tab

**Model:** Boltz-2 (open-source AF3-compatible structure predictor).
- URL: https://github.com/jwohlwend/boltz
- Install: `pip install boltz`

**Task:** Fine-tune Boltz-2 Pairformer with LoRA r=16 on TCR-pMHC complexes. Target: CDR3β RMSD < 1.5 Å on 30 held-out structures (current SOTA: 2.5 Å).

**Key steps:**
1. Parse PDB files → extract TCR alpha/beta chains + peptide + MHC chain
2. Compute CDR3β positions using IMGT numbering (positions 105-117)
3. Sequence identity split at <30% on CDR3β sequences (not full TCR)
4. Fine-tune with differential LR: Pairformer LR = 1e-5, head LR = 1e-4
5. Evaluate: superimpose predicted structure on experimental → compute CDR3β Cα RMSD

```python
# Boltz-2 fine-tuning entry point (pseudocode)
from boltz.model import Boltz2
model = Boltz2.from_pretrained('boltz2-weights')
inject_lora(model.pairformer, target_names=('W_q', 'W_k', 'W_v'), rank=16, alpha=32)
# Then use train_mutation_predictor-style loop with structure RMSD loss
```

## Mastery Check

On a first pass, success means you can:
1. explain the task, model adaptation strategy, and evaluation metric
2. say why LoRA is used here
3. distinguish conceptual understanding from implementation depth
4. identify what prerequisites you still need before a full reproduction


---
## 📊 SKEMPI v2 — Real ΔΔG Data

The SKEMPI v2 database contains 7,085 single-point mutations in protein-protein interfaces with experimentally measured ΔΔG values. This is the standard benchmark for stability and binding prediction.

**Critical note:** You MUST split by protein complex, not by mutation. Random splitting causes data leakage.


In [ ]:
# SKEMPI v2 DOWNLOAD AND PREPARATION
# SKEMPI: Structural Kinetics and Energetics of Mutant Protein Interactions

import urllib.request
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SKEMPI_URL = "https://life.bsc.es/pid/skempi2/database/download/skempi_v2.csv"
SKEMPI_FALLBACK_URL = "https://raw.githubusercontent.com/simonbray/skempi/main/skempi_v2.csv"

def load_skempi():
    """Download and parse SKEMPI v2."""
    for url in [SKEMPI_URL, SKEMPI_FALLBACK_URL]:
        try:
            with urllib.request.urlopen(url, timeout=15) as r:
                content = r.read().decode('utf-8')
            df = pd.read_csv(io.StringIO(content), sep=';')
            print(f"Downloaded SKEMPI v2: {len(df)} records")
            return df
        except Exception as e:
            print(f"  Failed ({url}): {e}")
    return None

skempi = load_skempi()

if skempi is None:
    # Simulate a SKEMPI-like dataset with realistic statistics
    print("\nGenerating simulated SKEMPI-like dataset with real statistics...")
    np.random.seed(42)
    n = 500
    proteins = [f'PDB_{i:04d}' for i in range(80)]  # 80 protein complexes
    complex_ids = np.random.choice(proteins, n)

    skempi_sim = pd.DataFrame({
        '#Pdb': complex_ids,
        'Mutation(s)_cleaned': [f'{np.random.choice(list("ACDEFGHIKLMNPQRSTVWY"))}A{np.random.randint(1,200)}G'
                                 for _ in range(n)],
        'ddG': np.random.normal(0.5, 2.0, n),   # kcal/mol, mean slightly destabilizing
        'Temperature': np.random.choice([25.0, 37.0], n),
        'pH': np.random.choice([7.0, 7.4], n),
    })
    skempi = skempi_sim
    IS_SIMULATED = True
    print(f"Simulated {n} mutations across {len(proteins)} protein complexes")
else:
    IS_SIMULATED = False
    # Standardize column names
    if 'Protein' in skempi.columns:
        skempi = skempi.rename(columns={'Protein': '#Pdb'})
    # Extract ΔΔG (may be in different columns depending on version)
    ddg_col = next((c for c in skempi.columns if 'ddG' in c or 'dG' in c), None)
    if ddg_col:
        skempi = skempi.rename(columns={ddg_col: 'ddG'})

print(f"\nSKEMPI summary:")
print(f"  Total mutations: {len(skempi)}")
if '#Pdb' in skempi.columns:
    n_proteins = skempi['#Pdb'].nunique()
    print(f"  Protein complexes: {n_proteins}")
if 'ddG' in skempi.columns:
    ddg = pd.to_numeric(skempi['ddG'], errors='coerce').dropna()
    print(f"  ΔΔG range: {ddg.min():.2f} to {ddg.max():.2f} kcal/mol")
    print(f"  Stabilizing (ΔΔG < -0.5): {(ddg < -0.5).sum()} ({(ddg < -0.5).mean()*100:.0f}%)")
    print(f"  Neutral (-0.5 to 0.5):    {((ddg >= -0.5) & (ddg <= 0.5)).sum()}")
    print(f"  Destabilizing (ΔΔG > 0.5): {(ddg > 0.5).sum()} ({(ddg > 0.5).mean()*100:.0f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(ddg, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
    axes[0].axvline(0, color='red', linestyle='--', label='ΔΔG = 0')
    axes[0].axvline(ddg.mean(), color='orange', linestyle='--', label=f'Mean = {ddg.mean():.2f}')
    axes[0].set_xlabel('ΔΔG (kcal/mol)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('SKEMPI v2: ΔΔG Distribution')
    axes[0].legend()

    if '#Pdb' in skempi.columns:
        per_protein = skempi.groupby('#Pdb').size()
        axes[1].hist(per_protein, bins=20, color='coral', edgecolor='white')
        axes[1].set_xlabel('Mutations per protein complex')
        axes[1].set_ylabel('Count')
        axes[1].set_title('Mutations per Protein Complex\n(skewed: some complexes are over-represented)')
    plt.tight_layout()
    plt.savefig('skempi_eda.png', dpi=100)
    plt.show()

## Protein-Level Data Splitting — Why Random Splits Fail

When predicting mutation effects (ΔΔG), a random 80/20 split leaks information: mutations from the *same protein* appear in both train and test, making the model look better than it really is on new proteins.

**The fix:** split by protein complex — all mutations from a given protein go entirely into either train or test, never both.  This tests true generalisation.

The code below demonstrates both the naive (wrong) and correct splitting strategies, then shows the accuracy gap between them.

In [ ]:
# CRITICAL: Protein-level data splitting (NOT random!)
# This is the most common mistake in ΔΔG prediction papers

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

print("DATA SPLITTING STRATEGY FOR SKEMPI")
print("=" * 55)
print()
print("WRONG: Random split (causes data leakage)")
print("  - Mutations from the SAME protein complex appear in both train and test")
print("  - Model learns protein-specific patterns, not generalizable features")
print("  - Inflated test performance: papers report ~0.85 r²")
print("  - Real-world performance on unseen proteins: ~0.5-0.6 r²")
print("  - This mistake has appeared in published papers!")
print()
print("CORRECT: Protein-level split")
print("  - All mutations from one protein complex go to either train or test")
print("  - Evaluates generalization to NEW protein complexes")
print("  - Honest performance: typically ~0.6-0.7 r²")
print()

if '#Pdb' in skempi.columns and 'ddG' in skempi.columns:
    df = skempi[['#Pdb', 'ddG']].copy()
    df['ddG'] = pd.to_numeric(df['ddG'], errors='coerce')
    df = df.dropna()

    proteins = df['#Pdb'].values
    ddg_vals = df['ddG'].values
    n = len(df)

    # Wrong: random split
    wrong_idx_tr = np.random.choice(n, int(0.8*n), replace=False)
    wrong_idx_te = np.setdiff1d(np.arange(n), wrong_idx_tr)
    wrong_proteins_in_both = set(proteins[wrong_idx_tr]) & set(proteins[wrong_idx_te])
    print(f"WRONG SPLIT: {len(wrong_proteins_in_both)} protein complexes appear in BOTH train and test")
    print(f"  (out of {df['#Pdb'].nunique()} total complexes)")

    # Correct: protein-level split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=proteins))
    correct_proteins_in_both = set(proteins[train_idx]) & set(proteins[test_idx])
    print(f"\nCORRECT SPLIT: {len(correct_proteins_in_both)} protein complexes in both")
    print(f"  Train: {len(train_idx)} mutations from {len(set(proteins[train_idx]))} complexes")
    print(f"  Test:  {len(test_idx)} mutations from {len(set(proteins[test_idx]))} complexes")

    # Simple baseline: mean ΔΔG prediction
    mean_train = ddg_vals[train_idx].mean()
    from sklearn.metrics import r2_score, mean_absolute_error
    r2_baseline = r2_score(ddg_vals[test_idx], np.full(len(test_idx), mean_train))
    mae_baseline = mean_absolute_error(ddg_vals[test_idx], np.full(len(test_idx), mean_train))
    print(f"\nBaseline (predict mean ΔΔG): R² = {r2_baseline:.3f}, MAE = {mae_baseline:.3f} kcal/mol")
    print("  Any model must beat this to be useful.")
    print()
    print("PRODUCTION DOWNLOAD COMMAND:")
    print("  wget 'https://life.bsc.es/pid/skempi2/database/download/skempi_v2.csv' \")
    print("       -O skempi_v2.csv")
    print("  # Then load: df = pd.read_csv('skempi_v2.csv', sep=';')")

## Loading ESM-2 and Embedding Protein Sequences

ESM-2 is a protein language model pretrained on 250 million sequences.  It maps a raw amino-acid string like `"MKTIIALSYIFCLVFA"` into a rich numerical embedding that captures evolutionary context, secondary structure propensity, and functional residue patterns — all without any structure data.

**What the code does:**
1. Loads a small ESM-2 checkpoint (`esm2_t6_8M_UR50D` — 8 M parameters).
2. Tokenises each sequence.
3. Runs a forward pass to extract per-residue embeddings.
4. Mean-pools over positions to get one fixed-size vector per protein.

These vectors become the input features for the ΔΔG regression head.

In [ ]:
# ESM-2 LOADING AND PROTEIN EMBEDDING
# ESM-2 is Meta AI's protein language model — free to use, available on HuggingFace

import torch
import numpy as np
import matplotlib.pyplot as plt

def load_esm2_and_embed(sequences, model_name='esm2_t6_8M_UR50D'):
    """
    Load ESM-2 (8M parameter version, runs on CPU) and embed protein sequences.
    Returns per-residue embeddings and sequence-level embeddings.
    """
    try:
        import esm
        model, alphabet = esm.pretrained.__dict__[model_name]()
        batch_converter = alphabet.get_batch_converter()
        model.eval()

        data = [(f'protein_{i}', seq) for i, seq in enumerate(sequences)]
        batch_labels, batch_strs, batch_tokens = batch_converter(data)

        with torch.no_grad():
            results = model(batch_tokens, repr_layers=[6], return_contacts=False)

        # Per-residue embeddings: shape (batch, seq_len, 320)
        embeddings = results['representations'][6]
        # Sequence-level: mean pool over residues (excluding BOS/EOS tokens)
        seq_embeds = embeddings[:, 1:-1, :].mean(dim=1)
        return embeddings, seq_embeds, True

    except ImportError:
        print("fair-esm not installed. Install with: pip install fair-esm")
        return None, None, False
    except Exception as e:
        print(f"ESM-2 loading failed: {e}")
        return None, None, False

# Protein sequences (chosen for diversity)
test_proteins = [
    ("Ubiquitin",      "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"),
    ("RAS_GTPase",     "MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSY"),
    ("p53_DBD",        "SVVVPYEPPEVGSDCTTIHYNYMCNSSCMGQMNRRPILTIITLEDSSGKLLGRNSFEVRVCACPGRDRRTEEENLRKKGE"),
    ("IgG1_VH",        "QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYGISWVRQAPGQGLEWMGWISAYNGNTNYAQKLQGRVTMTTDTSTSTAYMELRSLRSDDTAVYYCAR"),
]

print("Testing ESM-2 protein embedding...")
_, seq_embeds, success = load_esm2_and_embed([s for _, s in test_proteins])

if success and seq_embeds is not None:
    # Visualize embedding space
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    coords = pca.fit_transform(seq_embeds.numpy())

    fig, ax = plt.subplots(figsize=(7, 5))
    colors = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e']
    for i, (name, _) in enumerate(test_proteins):
        ax.scatter(coords[i, 0], coords[i, 1], s=200, color=colors[i],
                   zorder=5, label=name)
    ax.set_title('ESM-2 Protein Embeddings (PCA 2D)')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('esm2_embeddings.png', dpi=100)
    plt.show()
    print("ESM-2 embeddings visualized.")
else:
    print("\nSIMULATED ESM-2 OUTPUT (for illustration):")
    print()
    print("  When you load ESM-2, you get:")
    print("  - model: the transformer (8M to 15B parameters depending on variant)")
    print("  - alphabet: tokenizer mapping amino acids to integer IDs")
    print("  - repr_layers: which transformer layers to extract (last layer = best)")
    print()
    print("  Embedding shapes:")
    print("  - Per-residue: (batch_size, seq_len+2, embed_dim)")
    print("    +2 because of BOS/EOS tokens — slice [1:-1] to get residue embeddings")
    print("  - Sequence-level: mean pool → (batch_size, embed_dim)")
    print("    For ESM-2 8M: embed_dim=320")
    print("    For ESM-2 650M: embed_dim=1280")
    print()
    print("  TO RUN:")
    print("  pip install fair-esm")
    print("  Then: from esm import pretrained")
    print("        model, alphabet = pretrained.esm2_t6_8M_UR50D()")
    print("        # This downloads ~31MB model — runs on CPU in < 1 second per sequence")

## LoRA Rank Ablation — Choosing the Right Rank

LoRA rank `r` is the most important hyperparameter in adapter-based finetuning.

| rank | extra params | typical use case |
|------|-------------|-----------------|
| 1–2  | minimal     | extreme memory constraints |
| 4    | small       | most protein tasks (recommended default) |
| 8–16 | moderate    | multi-task or low-resource settings |
| 32+  | large       | approaching full-finetune quality |

**What the code does:** trains the same model with ranks 1, 2, 4, 8, 16 and plots Pearson r vs. parameter count so you can see the diminishing returns curve.

In [ ]:
# LoRA RANK ABLATION — Understand the rank-performance tradeoff
# Key question: what rank should I choose for LoRA?

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

class LoRALinear(nn.Module):
    """Linear layer with LoRA (Low-Rank Adaptation)."""
    def __init__(self, in_features, out_features, rank, alpha=16, dropout=0.0):
        """Initialise LoRA-adapted linear layer.

Args:
    in_features (int): input dimension.
    out_features (int): output dimension.
    rank (int): LoRA rank r.
    alpha (float): scaling constant; effective scale = alpha/rank.
    dropout (float): dropout applied before the B projection."""
        super().__init__()
        self.base = nn.Linear(in_features, out_features, bias=False)
        self.base.weight.requires_grad = False  # Frozen

        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)

        # Initialize: A ~ N(0,1), B = 0 (so initial ΔW = 0)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=np.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        """Apply the frozen linear weight plus the LoRA low-rank correction.

Args:
    x (Tensor): input tensor of shape (..., d_in).

Returns:
    Tensor: output of shape (..., d_out)."""
        return self.base(x) + self.scaling * self.lora_B(self.lora_A(self.dropout(x)))

    @property
    def n_trainable(self):
        """Return the number of trainable (LoRA adapter) parameters in this layer."""
        return sum(p.numel() for p in [self.lora_A.weight, self.lora_B.weight])

class ToyProteinModel(nn.Module):
    """Small transformer-like model for ΔΔG prediction with LoRA."""
    def __init__(self, rank=4, d_model=128):
        """Initialise toy protein model with LoRA-adapted projection layers.

Args:
    rank (int): LoRA rank applied to all linear layers.
    d_model (int): hidden representation dimension."""
        super().__init__()
        self.embed = nn.Embedding(21, d_model)  # 20 AA + padding
        self.layer1 = LoRALinear(d_model, d_model, rank=rank)
        self.layer2 = LoRALinear(d_model, 64, rank=rank)
        self.head = nn.Linear(64, 1)  # ΔΔG prediction

    def forward(self, x):
        """Apply the LoRA-adapted linear transformation.

Args:
    x (Tensor): input of shape (..., 10).

Returns:
    Tensor: output of shape (..., 5)."""
        h = self.embed(x).mean(1)
        h = torch.relu(self.layer1(h))
        h = torch.relu(self.layer2(h))
        return self.head(h).squeeze(-1)

# Generate synthetic ΔΔG-like regression data
def gen_ddg_data(n=400, seq_len=20):
"""Generate synthetic ΔΔG mutation data for rank-ablation experiments.

Args:
    n (int): number of synthetic mutations to generate.
    seq_len (int): sequence length (determines feature dimension).

Returns:
    Tuple[Tensor, Tensor]: (X features of shape (n, seq_len), y labels of shape (n, 1))."""
    seqs = torch.randint(0, 20, (n, seq_len))
    # True ΔΔG depends on mean embedding
    true_ddg = (seqs.float() - 10).mean(1) * 0.5 + torch.randn(n) * 0.5
    return seqs, true_ddg

X, y = gen_ddg_data(400)
X_tr, y_tr = X[:320], y[:320]
X_te, y_te = X[320:], y[320:]

# ── LoRA Rank Ablation ──
ranks = [1, 2, 4, 8, 16, 32, 64]
results_abl = []

for rank in ranks:
    model = ToyProteinModel(rank=rank)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())

    opt = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=1e-3
    )

    # Train for 30 epochs
    for ep in range(30):
        model.train()
        pred = model(X_tr)
        loss = nn.MSELoss()(pred, y_tr)
        opt.zero_grad()
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        val_mse = nn.MSELoss()(model(X_te), y_te).item()
        val_r2 = 1 - val_mse / y_te.var().item()

    results_abl.append({
        'rank': rank,
        'n_trainable': n_trainable,
        'pct_trainable': 100 * n_trainable / n_total,
        'val_mse': val_mse,
        'val_r2': val_r2,
    })

    print(f"  rank={rank:3d}: {n_trainable:6,} trainable params "
          f"({n_trainable/n_total*100:.1f}%) | val R²={val_r2:.3f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ranks_list = [r['rank'] for r in results_abl]
r2_list    = [r['val_r2'] for r in results_abl]
pct_list   = [r['pct_trainable'] for r in results_abl]

axes[0].semilogx(ranks_list, r2_list, 's-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('LoRA Rank (log scale)')
axes[0].set_ylabel('Validation R²')
axes[0].set_title('LoRA Rank Ablation: Performance vs Rank')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(ranks_list)
axes[0].set_xticklabels(ranks_list)

axes[1].plot(pct_list, r2_list, 's-', color='coral', linewidth=2, markersize=8)
for i, r in enumerate(results_abl):
    axes[1].annotate(f'r={r["rank"]}', (pct_list[i], r2_list[i]),
                     textcoords='offset points', xytext=(5, 3), fontsize=8)
axes[1].set_xlabel('% Trainable Parameters')
axes[1].set_ylabel('Validation R²')
axes[1].set_title('Performance vs Parameter Efficiency')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lora_rank_ablation.png', dpi=100)
plt.show()

print()
print("RANK SELECTION GUIDELINES:")
print("  Rank 1-2:  < 0.1% trainable params — very low capacity (task-specific adapters only)")
print("  Rank 4-8:  0.3-0.5% — sweet spot for most fine-tuning tasks")
print("  Rank 16:   ~1% — use when task is complex or training data is limited")
print("  Rank 32+:  diminishing returns; at rank=d_model, LoRA ≡ full fine-tuning")
print()
print("PUBLISHED FINDINGS:")
print("  LoRA original paper: rank=4 or 8 matches full fine-tuning on NLP tasks")
print("  AlphaFold fine-tuning (Ahdritz et al.): rank=16-32 for structure prediction")

---
## ⚠️ When Fine-Tuning Fails — Negative Results

Research is mostly failure. Here are the most common fine-tuning failure modes and how to detect them.


In [ ]:
# FINE-TUNING FAILURE MODES — Know These Before You Start

failure_guide = {
    "1. Domain Gap Too Large": {
        "description": "Pre-trained model was trained on very different data",
        "example": "Fine-tuning a language model pre-trained on English text for protein sequences",
        "symptoms": ["Training loss doesn't decrease past epoch 1-2",
                     "Val loss spikes after initial drop",
                     "Output predictions are near-constant"],
        "detection": "Plot gradient norms — if all near zero, features aren't activating",
        "fixes": ["Use a domain-specific pre-trained model (ESM-2 for proteins, not GPT-2)",
                  "Reduce learning rate by 10-100×",
                  "Only fine-tune the last 2 layers initially"],
    },
    "2. Catastrophic Forgetting": {
        "description": "Fine-tuning destroys pre-trained representations",
        "example": "Fine-tuning ESM-2 for ΔΔG with LR=1e-3 (too high) — loses secondary structure info",
        "symptoms": ["Val loss improves, then INCREASES after many epochs",
                     "Model loses performance on original pre-training tasks",
                     "Attention maps become random (if visualized)"],
        "detection": "Evaluate on a held-out pre-training validation task periodically",
        "fixes": ["Use smaller LR for pre-trained layers (e.g., 1e-5 vs 1e-3 for head)",
                  "Use LoRA — only adapts small ΔW, original weights frozen",
                  "Add L2 regularization toward initialization (EWC)"],
    },
    "3. Insufficient Data": {
        "description": "Not enough labeled examples for the task",
        "example": "Fine-tuning on 50 ΔΔG measurements — model memorizes all of them",
        "symptoms": ["Train loss → near 0, val loss stays high",
                     "Val R² oscillates wildly (high variance)",
                     "Performance sensitive to random seed"],
        "detection": "If train/test performance correlation with seed > 0.3, data is too small",
        "fixes": ["Use LoRA rank 1-2 (minimal parameters = minimal overfitting)",
                  "Use cross-validation (n_splits ≥ 5) for evaluation",
                  "Semi-supervised: add unlabeled data with pseudo-labels"],
    },
    "4. Wrong Task Head": {
        "description": "Output head architecture doesn't match task requirements",
        "example": "Using sigmoid + BCE loss for ΔΔG regression (should be linear + MSE)",
        "symptoms": ["Loss doesn't decrease at all",
                     "Predictions are all the same value",
                     "NaN loss appears early"],
        "detection": "Check: gradient of loss w.r.t. head parameters — if zero, head is wrong",
        "fixes": ["Regression: linear head + MSE (or Huber for outliers)",
                  "Binary classification: sigmoid + BCE",
                  "Multi-class: softmax + CrossEntropy"],
    },
    "5. Learning Rate Issues": {
        "description": "LR too high (unstable) or too low (slow, local minimum)",
        "example": "LR=1e-2 on a pre-trained transformer → NaN in first batch",
        "symptoms": ["NaN loss: LR too high or gradient explosion",
                     "Loss decreases by < 0.001 per epoch: LR too low",
                     "Oscillating loss curves: LR at edge of stability"],
        "detection": "Run LR finder: increase LR exponentially, plot loss — use LR just before divergence",
        "fixes": ["Warmup + cosine decay schedule",
                  "Gradient clipping (max_norm=1.0)",
                  "Layer-wise LR decay: outer layers 1×, inner layers 0.1×"],
    },
}

print("FINE-TUNING FAILURE MODE GUIDE")
print("=" * 60)
for i, (name, info) in enumerate(failure_guide.items(), 1):
    print(f"\n{'─'*60}")
    print(f"FAILURE MODE {name}")
    print(f"  Description: {info['description']}")
    print(f"  Example: {info['example']}")
    print(f"  Symptoms:")
    for s in info['symptoms']:
        print(f"    • {s}")
    print(f"  Detect: {info['detection']}")
    print(f"  Fixes:")
    for f in info['fixes']:
        print(f"    → {f}")

print()
print("GOLDEN RULE: Run a diagnostic pass BEFORE full fine-tuning:")
print("  1. Overfit on 10 samples → confirms forward pass works, loss decreases")
print("  2. Train for 2 epochs → confirms gradients flow, LR is reasonable")
print("  3. Evaluate on 1 held-out example → confirms output format is correct")
print("  If any of these 3 steps fail → fix before training for real.")

## 🐛 Debug Exercise — LoRA Fine-Tuning Setup

Find and fix the **3 bugs** in the LoRA fine-tuning configuration below.

**Expected correct output:**
```
LoRA parameter count < full layer parameter count
Learning rate for LoRA: ~1e-4 (10x lower than full fine-tuning ~1e-3)
Train/test split: grouped by protein complex, not random
```

<details>
<summary>Show answer</summary>

**Bug 1 — LoRA rank too high:** LoRA with `r=64` on a layer with `dim=32` means the
low-rank matrices A (32×64) and B (64×32) together have MORE parameters than the original
weight (32×32). This defeats the purpose of LoRA. Fix: `r <= dim // 4`, e.g., `r=8`.

**Bug 2 — Learning rate too high for LoRA:** LoRA adapts pre-trained weights with small
updates. Using the same LR as full fine-tuning (1e-3) causes instability. Fix: use 1e-4
or lower for LoRA parameters.

**Bug 3 — ΔΔG split by random:** If the same protein complex appears in both train and test,
the model memorizes complex-specific features. Fix: group by `complex_id` and split groups,
ensuring no complex is in both sets.
</details>


In [ ]:
# DEBUG EXERCISE — LoRA fine-tuning configuration, find and fix 3 bugs
import torch
import torch.nn as nn
import numpy as np

# --- Bug 1: LoRA rank larger than model dimension ---
class LoRALayer(nn.Module):
    """LoRA adapter used in the debug exercise. See LoRALayer in cell 11 for the full implementation."""
    def __init__(self, in_dim, out_dim, r):
        """Initialise the LoRA adapter (debug exercise version).

Args:
    in_dim (int): input feature dimension.
    out_dim (int): output feature dimension.
    r (int): LoRA rank."""
        super().__init__()
        self.A = nn.Linear(in_dim, r, bias=False)
        self.B = nn.Linear(r, out_dim, bias=False)
        self.scale = 1.0 / r

    def forward(self, x):
        """Apply the frozen linear weight plus the LoRA low-rank correction.

Args:
    x (Tensor): input tensor of shape (..., d_in).

Returns:
    Tensor: output of shape (..., d_out)."""
        return self.B(self.A(x)) * self.scale

model_dim = 32
# BUG 1: r=64 > model_dim=32 — LoRA has MORE params than the original weight!
r = 64     # should be <= model_dim // 4, e.g., r=8
lora = LoRALayer(model_dim, model_dim, r)
lora_params = sum(p.numel() for p in lora.parameters())
full_params = model_dim * model_dim   # equivalent full weight matrix
print(f"LoRA params (r={r}): {lora_params}")
print(f"Full weight params: {full_params}")
print(f"LoRA is {'LARGER' if lora_params > full_params else 'smaller'} than full layer — BUG!")

# --- Bug 2: learning rate too high for LoRA ---
# BUG 2: LoRA should use 10x lower LR than full fine-tuning
full_finetune_lr = 1e-3
lora_lr = 1e-3        # BUG: same as full fine-tuning — should be 1e-4
print(f"\nLoRA learning rate: {lora_lr}  (same as full fine-tuning — too high!)")
print(f"Recommended LoRA LR: {full_finetune_lr / 10}  (10x lower)")

# --- Bug 3: ΔΔG train/test split ignores protein complex ---
np.random.seed(42)
n_mutations = 200
# Each mutation belongs to one of 10 protein complexes
complex_ids = np.repeat(np.arange(10), 20)  # 20 mutations per complex
delta_delta_g = np.random.randn(n_mutations)

# BUG 3: random split — same complex may appear in both train and test
from sklearn.model_selection import train_test_split
idx = np.arange(n_mutations)
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42)  # random!

train_complexes = set(complex_ids[train_idx])
test_complexes  = set(complex_ids[test_idx])
leaked = train_complexes & test_complexes
print(f"\nComplexes in both train AND test (data leakage): {len(leaked)}")
print(f"Leaked complex IDs: {sorted(leaked)}")
print("Fix: use GroupShuffleSplit with groups=complex_ids to keep complexes separate")


## Notebook Complete

**You can now:**
- Fine-tune a Pairformer-based protein structure model with LoRA adapters
- Apply the complete fine-tuning loop: data loading, loss, validation, and checkpointing

**Where this fits:**
- Previous: 00_openfold3_walkthrough
- Next: 11_membrane_protein_dynamics/01_membrane_protein_dynamics — Module 11 — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes